# 3.0 - Modelado

## Telco Customer Churn

Entrenamos y comparamos 4 clasificadores (Logistic Regression como
*baseline*, Random Forest, XGBoost y LightGBM) reutilizando el preprocesador
ajustado en el paso 2.

**Estrategia:**
- Extraemos una porción de **validación** del set de entrenamiento y
  optimizamos el **umbral por F1** sobre validación (nunca sobre test).
- Desbalance (73/27): `class_weight`/`scale_pos_weight`. SMOTE solo como
  refuerzo si no se alcanza F1 >= 0.60.
- Registramos cada corrida en **MLflow** (params, métricas, artefactos).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent))

from pprint import pprint

from src.models.train_model import build_models
from src.models.predict_model import predict_proba
from src.features.build_features import load_processed_data

In [2]:
X_train, X_test, y_train, y_test = load_processed_data()
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")
print(f"\nDistribución y_train:\n{y_train.value_counts(normalize=True)}")

X_train: (5634, 19) | X_test: (1409, 19)
y_train: (5634,) | y_test: (1409,)

Distribución y_train:
Churn
No     0.734647
Yes    0.265353
Name: proportion, dtype: float64


In [3]:
for name, (factory, params) in build_models().items():
    print(name, "->", params)

logistic_regression -> {}
random_forest -> {}
xgboost -> {}
lightgbm -> {}


## Entrenamiento y evaluación

Entrenamos los cuatro modelos, elegimos el umbral por F1 sobre la validación
y evaluamos sobre **test**. El mejor modelo por F1 en test se persiste en
`models/best_model.joblib`.

In [4]:
from src.models.train_model import train_and_evaluate

results = train_and_evaluate()

[logistic_regression] Val-F1=0.636 | Test-F1=0.617 | Test-Recall=0.727 | AUC=0.843 | thresh=0.57


[random_forest] Val-F1=0.615 | Test-F1=0.594 | Test-Recall=0.687 | AUC=0.820 | thresh=0.43


[xgboost] Val-F1=0.623 | Test-F1=0.623 | Test-Recall=0.767 | AUC=0.837 | thresh=0.46


[lightgbm] Val-F1=0.620 | Test-F1=0.607 | Test-Recall=0.684 | AUC=0.826 | thresh=0.52


In [5]:
print("\n=== Métricas sobre TEST por modelo ===")
for name, res in results.items():
    print(f"\n[{name}] umbral={res['threshold']:.2f}")
    pprint(res["metrics"])
    print("Matriz de confusión:")
    print(res["cm"])


=== Métricas sobre TEST por modelo ===

[logistic_regression] umbral=0.57
{'accuracy': 0.7608232789212207,
 'auc_roc': 0.8425792451367898,
 'f1': 0.6174801362088536,
 'precision': 0.5364891518737672,
 'recall': 0.7272727272727273}
Matriz de confusión:
            Pred No  Pred Yes
Actual No       800       235
Actual Yes      102       272

[random_forest] umbral=0.43
{'accuracy': 0.7501774308019872,
 'auc_roc': 0.8198933064661965,
 'f1': 0.5935334872979214,
 'precision': 0.5223577235772358,
 'recall': 0.6871657754010695}
Matriz de confusión:
            Pred No  Pred Yes
Actual No       800       235
Actual Yes      117       257

[xgboost] umbral=0.46
{'accuracy': 0.7537260468417317,
 'auc_roc': 0.8369048541682814,
 'f1': 0.6232356134636265,
 'precision': 0.5246800731261426,
 'recall': 0.767379679144385}
Matriz de confusión:
            Pred No  Pred Yes
Actual No       775       260
Actual Yes       87       287

[lightgbm] umbral=0.52
{'accuracy': 0.7650816181689141,
 'auc_roc': 0

## Comprobación de umbrales mínimos

El modelo ganador (XGBoost) debe superar los mínimos sobre test:
AUC >= 0.80, F1 >= 0.60 y Recall >= 0.60.

In [6]:
from src.models.train_model import save_artifacts, log_to_mlflow, MIN_F1_THRESHOLD

best_name = save_artifacts(results)
log_to_mlflow(results, build_models())
best = results[best_name]["metrics"]
print(f"\nMejor modelo: {best_name}")
print(f"Test-F1={best['f1']:.3f} | Recall={best['recall']:.3f} | AUC={best['auc_roc']:.3f}")
assert best["f1"] >= MIN_F1_THRESHOLD
assert best["auc_roc"] >= 0.80
assert best["recall"] >= 0.60
print("Umbrales mínimos cumplidos.")

Mejor modelo: xgboost
Umbral óptimo (F1 valid): 0.460
Guardado en: D:\DSRP-ML2-Proyecto-Final\models\best_model.joblib


2026/09/01 22:55:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/09/01 22:55:28 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/09/01 22:55:34 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/09/01 22:55:45 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



Mejor modelo: xgboost
Test-F1=0.623 | Recall=0.767 | AUC=0.837
Umbrales mínimos cumplidos.


## Inferencia sobre un lote (test)

Usamos `predict_model` para reproducir la predicción con el modelo y el
umbral óptimos persistidos.

In [7]:
from sklearn.metrics import f1_score

out = predict_proba(X_test)
out.head(10)

,probability,prediction
0,0.024390,0
1,0.956244,1
2,0.251015,0
3,0.453277,0
4,0.007187,0
5,0.854457,1
6,0.812600,1
7,0.132329,0
8,0.008564,0
9,0.579533,1


In [8]:
y_test_num = (y_test == "Yes").astype(int)
print(f"F1 (test) reconstruda: {f1_score(y_test_num, out['prediction']):.3f}")
print(f"Tasa de positivos predicha: {out['prediction'].mean():.3f}")

F1 (test) reconstruda: 0.623
Tasa de positivos predicha: 0.388
